# E1 - Andor EMCCD photon counting over time

Photon counting on the Andor EMCCD: 90 s frames (4x4 binning, EM gain 300, 1 MHz, deep cooling) back to back until the leaf dries out, then a threshold at 5 sigma of the read noise turns each frame into photon events. This is the reference for everything else: the photon budget over time (events per pixel per second), a 90 s time resolution, and an event map that shows where the emission comes from.

Why 90 s (David): with ~1e-4 e-/px/s dark current at -95 °C, the dark charge of a frame catches up with the clock-induced charge after roughly a minute. Shorter frames add CIC without gaining anything, longer frames only coarsen the time resolution.

**Check after the first few frames**: run the analysis cells at the bottom once (ROIs may stay empty). If the summary warns that more than 20 % of the pixels fire per frame, thresholding undercounts: stop, shorten `EXPOSURE_S` (or use 2x2 binning) and start again.

**Before you start**

- Nothing in this notebook moves the stage, the focus or switches a light. Position and focus the sample by hand, close the enclosure, switch the illumination off.
- Run once with `DRY_RUN = True` (three 5 s frames through the whole chain), then set it to `False`.
- The camera settings are applied by property name. A wrong name stops the notebook before anything changes and prints the names the camera has.
- Andor snaps go through `MMCoreController.snapMMCoreToDisk`, the same path as the ImSwitch Andor panel: one exposure at a time, cancellable, temperature readable while it integrates.

## 1 Connect

In [ ]:
from pathlib import Path
import biophoton as bp

# None = $IMSWITCH_API_URL, which ImSwitch sets for the notebooks it serves itself.
# From another computer use the address you open in the browser, e.g. "http://192.168.1.20/imswitch/api".
IMSWITCH_URL = None
client = bp.connect(IMSWITCH_URL)
print("Connected to", client.base_uri, "| detectors:", client.settingsManager.getDetectorNames())

## 2 Settings

In [ ]:
DETECTOR = "Andor"           # detector name in the setup JSON (FRAME_6_ANDOR.json)
RUN_NAME = "cress_h2o2"
DRY_RUN = True               # True: 3 short frames through the whole chain, marked as dry run

SAMPLE = {
    "species": "Lepidium sativum (garden cress)",
    "treatment": "injured + 2 % H2O2",
    "prepared_at": None,     # ISO time of injury + H2O2, e.g. "2026-10-23T09:40"; plots count from here
    "notes": "",
}

EXPOSURE_S = 90                 # one frame, back to back; see the check above
DURATION_H = 12              # no new frame starts after this; stop earlier with the Stop cell
CAMERA = {                   # Andor MMCore property names, applied in this order
    "ReadoutMode": "1.000 MHz",
    "Binning": "4",              # 4x4: 52 µm super-pixels, 16x the charge per pixel
    "Gain": 300,                 # EM gain, as in the June data (above 300 ages the EM register faster)
    "CCDTemperatureSetPoint": -95,
}
COOLDOWN = {"tolerance_c": 2.0, "timeout_s": 45 * 60}
ENV_PERIOD_S = 30
# Also log the other camera, e.g. [{"label": "toupcam", "url": "http://192.168.1.21/imswitch/api", "detector": None}]
OTHER_SERVERS = []
DATA_ROOT = Path("data")     # run folders go to data/E1_andor_photon_counting/<date>_<RUN_NAME>/

N_FRAMES = None
if DRY_RUN:
    EXPOSURE_S, DURATION_H, N_FRAMES = 5, None, 3

STEPS = [{"label": "frame", "exposure_ms": EXPOSURE_S * 1000, "n_frames": N_FRAMES,
          "duration_s": DURATION_H * 3600 if DURATION_H else None}]
print(f"{'DRY RUN: ' if DRY_RUN else ''}{EXPOSURE_S:g} s frames, "
      f"{f'{N_FRAMES} frames' if N_FRAMES else f'for {DURATION_H} h'} on {DETECTOR}")

## 3 Run folder, camera settings, environment log, cooldown

Creates a new run folder (never reuses one), applies `CAMERA` by name and reads it back, starts the environment log and waits for the sensor to reach its set point.

In [ ]:
try:
    if acq.running:
        raise RuntimeError("An acquisition is still running in this kernel - stop it first (Stop cell)")
except NameError:
    pass

run = bp.Run("E1_andor_photon_counting", RUN_NAME, DATA_ROOT)
readback = bp.apply_settings(client, DETECTOR, CAMERA)
run.log(f"camera read back: {readback}")
run.update_config(
    kind="timecourse", experiment="E1", title="E1 - Andor EMCCD photon counting over time",
    description='Photon counting on the Andor EMCCD: 90 s frames (4x4 binning, EM gain 300, 1 MHz, deep cooling) back to back until the leaf dries out, then a threshold at 5 sigma of the read noise turns each frame into photon events. This is the reference for everything else: the photon budget over time (events per pixel per second), a 90 s time resolution, and an event map that shows where the emission comes from.',
    sample=SAMPLE, server=client.base_uri, detector=DETECTOR, dry_run=DRY_RUN,
    camera_settings=CAMERA, camera_readback=readback, steps=STEPS,
    analysis={"mode": "photon_counting", "label": "frame", "nsigma": 5.0, "window_min": 10},
    rois={"sample": None, "background": None})
run.save_json("camera_start.json", bp.parameters(client, DETECTOR)[1])

env = bp.EnvLogger(run.dir / "environment.csv",
                   bp.env_sources(client, DETECTOR, "andor", OTHER_SERVERS), ENV_PERIOD_S).start()
bp.wait_for_temperature(client, DETECTOR, CAMERA["CCDTemperatureSetPoint"], log=run.log, **COOLDOWN)

## 4 Start

Runs in a background thread and returns at once.

In [ ]:
acq = bp.Acquisition(run, client, DETECTOR, STEPS, backend="mmcore", env=env,
                     env_label="andor").start()

## 5 Dashboard

Redraws from the files on disk every 15 s. Interrupting it (■) stops only the display; re-run it any time, e.g. after the browser reconnects.

In [ ]:
bp.dashboard(run.dir, acq, refresh_s=15)

## Stop

Interrupt the dashboard cell first (■): the kernel runs one cell at a time. This cancels the running
exposure on the server. Frames already saved stay on disk.

In [ ]:
acq.stop()
print(acq.state)

## Finish

Stops the environment log and stores the camera state at the end. Run it after the acquisition has
finished or was stopped.

In [ ]:
env.stop()
run.save_json("camera_end.json", bp.parameters(client, DETECTOR)[1])
run.log("run finished")

## Analysis and report

Works on any run folder: point `RUN_DIR` at an older run to redraw it. Everything is read from the
files in that folder and written back to it (`figures/`, `analysis.csv`, `summary.json`,
`README.md`), so re-running these cells never loses data.

1. Show a frame and read off a **sample** ROI on the leaf and a **background** ROI next to it, as
   `[x, y, width, height]` in pixels of the saved frame.
2. Store the ROIs and redraw.

In [ ]:
RUN_DIR = run.dir           # or e.g. Path("data/E1_andor_photon_counting/<run folder>")
bp.show_frame(RUN_DIR, index=-1);

In [ ]:
bp.set_rois(RUN_DIR, {"sample": None, "background": None})   # e.g. {"sample": [80, 90, 60, 60], "background": [10, 10, 40, 40]}
summary = bp.redraw(RUN_DIR, show=True)
summary

The run folder now holds `README.md` with this summary and the figures, and `report.ipynb`, which
redraws them from that folder alone (it imports the copy of `biophoton.py` stored with the run).